# v20: あなたのGPU PF 出力を LB7.191 ノートに「合わせる」ブリッジ

**目的**: 7.191 ノートの PF(CPU/numba)を、あなたの **v20 GPU PF(=submit_v20 と同じエンジン)** の出力に差し替える。
PFパラメータはあなたのもの(最適)を使い、出力フォーマットだけ 7.191 に合わせる。

## 差し替える 7.191 の関数(同じ戻り値にする)
- `run_pf_ancc(hw,tw_tvt,tw_gr,N)` → `(pf_a, std_a)`  … GBM特徴用(ANCC PF)
- `run_pf_lik_ensemble_scales(hw,tw,scales=(3,5,8,12),n_seeds=128)` → `{pf_scale_3/5/8/12, pf_mean}` … selector用

## 鍵: GPU 1回で multi-scale を作る
GPU PF は per-seed 軌跡 `pts(S,T)` と `log_lik(S)` を持つ。**スケール X ごとに `softmax(log_lik/X)` で seed を再加重**するだけで
`pf_scale_X` を一発生成(7.191 の 128seed CPU 再走行を GPU 1回に圧縮)。`v21_common.run_forward_scales` がそれ。

## 安全性(重要)
- **selector PF(`run_pf_lik_ensemble_scales`)の差し替えは安全**(selectorは決定的ブレンドで学習モデルを介さない)。
- **GBM特徴の `run_pf_ancc` 差し替えは注意**: 事前学習GBM/fleongg は「7.191 own PF の特徴分布」で学習済み。
  あなたのPFに替えると特徴がズレて**逆効果の恐れ**。→ まずは **selector のみ差し替え** か、**最終ブレンドにあなたのPF予測を1メンバー追加**が安全。
- `run_pf_z` は据え置き(CPU)。
- 評価はノイズ床±0.2 / 硬い核 per-well で。

## 環境
- このノートは**ローカル検証用**(`import v21_common`、`SPLIT='train'` で真値比較可)。
- **Kaggleの7.191に入れる**ときは、v21_common は使えない(パス固定)ので **submit_v20 のインラインGPU PF + `run_forward_scales` の再加重 + 下の drop-in** を 7.191 に貼る(下のセル参照)。

In [2]:
import sys, os, glob, threading, time, numpy as np
sys.path.insert(0, "../v21")
import importlib, v21_common as C; importlib.reload(C)

# ===== 設定(7.191 と一致させる)=====
SPLIT    = "train"            # ローカル検証=train(真値あり) / Kaggle推論=test
SCALES   = (3.0, 5.0, 8.0, 12.0)   # 7.191 の SELECTOR_SCALES と一致
N_SEEDS  = 32                # ★あなたのPFの調整regime(submit_v20=32)。7.191自身のPFは128だが流用しない
CHUNK    = 24                # VRAMで調整
USE_PHYS = True              # あなたのバンクPF(r0=use_phys)に合わせる
N_WELLS  = 60                # ローカル検証はサブセットで可(None=全部)。硬い核を別途追加して easy/hard で分けて見る

ns = C.load_v9_ns()
p  = C.r0_params(ns, use_phys=USE_PHYS, n_seeds=N_SEEDS)   # ★"あなたのPF"。別paramにするならここを差し替え
def list_wells(split):
    return sorted(os.path.basename(q).split("__")[0] for q in glob.glob(str(C.DATA_DIR/split/"*__horizontal_well.csv")))
wells = list_wells(SPLIT)
if N_WELLS: wells = wells[:N_WELLS] + [w for w in C.HARD_WELLS if w not in wells[:N_WELLS]]
print("wells:", len(wells), "| device:", C.DEVICE)

[v9] exec 中(FI/DI fit を含む)...
PF parameter sets:
  pf_1: suffix=_1, N=600, seeds=32, jump=(0.002, 5.0)
  pf_2: suffix=_2, N=600, seeds=32, jump=(0.0, 10.0)
  pf_3: suffix=_3, N=600, seeds=32, jump=(0.002, 5.0)
[v21] r0 params: r0 N=500 seeds=32 phys=on
wells: 79 | device: cuda


In [3]:
# ===== GPU PF を1回走らせて multi-scale 出力をキャッシュ(wid別)=====
inps, names = [], []
for wid in wells:
    x = C.build_inputs(wid, ns, p, split=SPLIT)
    if x is None: continue
    inps.append(x); names.append(wid)
t0 = time.perf_counter()
res = C.run_forward_scales(inps, p, N_SEEDS, C.DEVICE, C.SEED, CHUNK, scales=SCALES)
CACHE = {wid: d for wid, d in zip(names, res)}
print(f"GPU PF cached: {len(CACHE)} wells in {time.perf_counter()-t0:.1f}s ({C.DEVICE})")
print("keys:", list(next(iter(CACHE.values())).keys()))

GPU PF cached: 79 wells in 39.7s (cuda)
keys: ['pf_scale_3', 'pf_scale_5', 'pf_scale_8', 'pf_scale_12', 'pf_ancc', 'std', 'pf_mean']


In [4]:
# ===== drop-in: 7.191 の PF 関数と同じ戻り値 =====
_TLS = threading.local()
def set_well(wid):
    """7.191 の各坑井処理の直前に呼ぶ(wid を解決するため)。"""
    _TLS.wid = wid

def gpu_run_pf_ancc(hw, tw_tvt, tw_gr, N=None):
    d = CACHE.get(getattr(_TLS, "wid", None))
    if d is None: return np.array([]), np.array([])
    return d["pf_ancc"].astype(np.float64), d["std"].astype(np.float64)

def gpu_pf_by_scale(hw, tw, scales=SCALES, n_particles=500, n_seeds=128):
    d = CACHE.get(getattr(_TLS, "wid", None))
    if d is None:
        raise RuntimeError("set_well(wid) を先に呼んでください")
    out = {k: v for k, v in d.items() if k.startswith("pf_scale_")}
    out["pf_mean"] = d["pf_mean"]
    return out

print("drop-ins ready: set_well / gpu_run_pf_ancc / gpu_pf_by_scale")

drop-ins ready: set_well / gpu_run_pf_ancc / gpu_pf_by_scale


In [5]:
# ===== ローカル検証: 各スケールを easy / hard / all に分けて RMSE(真値, SPLIT=train のみ)=====
# 注意: all は検証セットに硬い核を入れている割合で変わる(tail支配)。engine健全性は easy で見る。
if SPLIT == "train":
    def rmse(parts):
        e = np.concatenate(parts); e = e[np.isfinite(e)]; return float(np.sqrt(np.mean(e**2))) if len(e) else float('nan')
    KEYS = [f"pf_scale_{s:g}" for s in SCALES] + ["pf_ancc", "pf_mean"]
    acc = {k: {"easy": [], "hard": []} for k in KEYS}
    for wid in names:
        x = C.build_inputs(wid, ns, p, split="train")
        if x is None or x["tvt_true"] is None: continue
        grp = "hard" if wid in C.HARD_WELLS else "easy"
        for k in KEYS: acc[k][grp].append(CACHE[wid][k] - x["tvt_true"])
    ne = sum(w not in C.HARD_WELLS for w in names); nh = sum(w in C.HARD_WELLS for w in names)
    print(f"easy坑井={ne}  hard坑井={nh}\n")
    print(f"{'output':>12s} {'easy':>8s} {'hard':>8s} {'all':>8s}")
    for k in KEYS:
        e = rmse(acc[k]["easy"]) if acc[k]["easy"] else float('nan')
        h = rmse(acc[k]["hard"]) if acc[k]["hard"] else float('nan')
        a = rmse(acc[k]["easy"] + acc[k]["hard"])
        print(f"{k:>12s} {e:8.3f} {h:8.3f} {a:8.3f}")
    print("\n※ easy が一桁(~9前後)なら engine 正常。all が高いのは検証セットに硬い核を多く入れているため(tail支配)。")
    print("  r0 の記録: overall_on_all≈9.17(全773井, 32seed)。")
else:
    print("SPLIT=test のため真値評価はスキップ(Kaggle推論用)")

easy坑井=59  hard坑井=20

      output     easy     hard      all
  pf_scale_3    7.897   29.687   16.359
  pf_scale_5    7.530   28.743   15.797
  pf_scale_8    7.326   28.260   15.504
 pf_scale_12    7.290   28.117   15.426
     pf_ancc    7.410   28.463   15.626
     pf_mean    7.450   28.616   15.711

※ easy が一桁(~9前後)なら engine 正常。all が高いのは検証セットに硬い核を多く入れているため(tail支配)。
  r0 の記録: overall_on_all≈9.17(全773井, 32seed)。


## 7.191 ノートへの差し込み手順(monkeypatch)

このノート(セル1〜3)を 7.191 ノートの**特徴生成/selectorの直前**に挿入し、以下を行う:

```python
# 1) 上の bridge セルを実行して CACHE と drop-in を用意(SPLIT='test')

# 2) selector PF を GPU 版へ差し替え(安全)
run_pf_lik_ensemble_scales = gpu_pf_by_scale          # 同名で上書き

# 3) 各坑井ループで wid をセット(selectorループ内、PF呼び出しの前)
for wid in test_wids:
    set_well(wid)                                     # ★この1行を追加
    hw_te = ...                                        # 既存
    pf_by_scale = run_pf_lik_ensemble_scales(hw_te, tw_ref, n_seeds=128)
    ...

# 4)(任意・要検証)GBM特徴の ANCC PF も GPU へ。事前学習モデルとの整合に注意:
# run_pf_ancc = gpu_run_pf_ancc      # build_well の前に set_well(wid) が必要
```

### 推奨の進め方
1. **まず selector PF だけ GPU 化**(3 のみ)。selector は決定的ブレンドなので学習モデルを壊さない。速度↑＆あなたのPF採用。
2. 効果を**硬い核 per-well** と CV で確認(ノイズ床±0.2)。
3. GBM特徴の置換(4)は、事前学習GBM/fleongg を**自分で再学習する場合のみ**(さもないと特徴分布ズレで悪化しうる)。
4. 別案: あなたのPF予測(`pf_scale_8` 等)を**最終ブレンドに1メンバー追加**(純粋な足し算で最も安全)。

### Kaggle で v21_common が使えない場合
`submit_v20.ipynb` のインライン GPU PF(`_pf_core`/`_pad_wells`/build_pf_inputs)＋ `run_forward_scales` の再加重ロジック(このノートの元 = v21_common)を 7.191 にコピーして、同じ drop-in を定義する。